In [ ]:
### Cu 003 processing ###


#%% load the packages
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.ticker import FuncFormatter
from matplotlib import cm
import matplotlib as mpl
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import defdap.hrdic as hrdic
import defdap.ebsd as ebsd
import defdap.experiment as experiment
from defdap.quat import Quat
from defdap.plotting import MapPlot

from pathlib import Path

import copy 
import pandas as pd
import datetime

from scipy.signal import find_peaks
from scipy.linalg import sqrtm, polar
from scipy import stats
from scipy import interpolate 
from scipy import ndimage


import skimage as ski


import os

# get dictools stuff 
import sys
# sys.path.append("c:/work/hrdic-tools/")
# import dictools

plt.rcParams['svg.fonttype'] = 'none'

%matplotlib qt

In [ ]:
def calc_rotations(dic_map):
    # calculate rotations from dic displacement field 

    # extract deformation gradient
    f = dic_map.data.f

    # calculate rotation as ang = (F21 - F12)/2
    rot = (f[0,1,:,:] - f[1,0,:,:])/2

    # centre on mean 
    rot = rot - np.nanmean(rot)
    
    return rot 

In [ ]:

exp = experiment.Experiment()

# load DIC data 
data_dir = Path('./DIC/pyvale/')
dic_frame = experiment.Frame()

# dic_step_list = sorted(data_dir.glob('dic_output_fixed_Step_*_subset_31_step_10.csv'))

for dic_file in sorted(data_dir.glob('dic_output_fixed_Step_*_subset_31_step_10.csv')):
    hrdic.Map(dic_file, experiment=exp, frame=dic_frame,data_type='pyvale-csv')

# dic_file = dic_step_list[-2]
# hrdic.Map(dic_file, experiment=exp, frame=dic_frame,data_type='pyvale-csv')

hfw = 20.0 # microns
pixelwidth = 2048
pixelsize = hfw/pixelwidth



# calculate rotations    
for inc, dic_map in exp.iter_over_maps('hrdic'):
    # add rotation map to dic

    # calculate rotation
    rot = calc_rotations(dic_map)*180/np.pi

    # add to dic_map
    dic_map.data.add(
        'r_ang', rot,
        unit='°', type='map', order=0,
        plot_params={
            'plot_colour_bar': True,
            'clabel': 'Rotation',
            'cmap': 'RdBu_r'
        }
    )



for inc, dic_map in exp.iter_over_maps('hrdic'):
    dic_map.set_scale(pixelsize)
    dic_map.set_crop(left=100,right=100,top=100,bottom=100)
    # dic_map.plot_map('max_shear',vmin=0,vmax=0.01,plot_scale_bar=True)
    print(dic_map)

In [ ]:
# sign conventions for defdap

# coordinates 
fig,ax = plt.subplots(1,2)
ax[0].imshow(dic_map.data.coordinate[0,:,:])
ax[0].set_title('x - coordinate')
ax[1].imshow(dic_map.data.coordinate[1,:,:])
ax[1].set_title('y - coordinate')
plt.tight_layout()

# displacements 
vmin = -600
vmax = -vmin
fig,ax = plt.subplots(1,2)
ax[0].imshow(dic_map.data.displacement[0,:,:],cmap='RdBu_r',vmin=vmin,vmax=vmax)
ax[0].set_title('u - displacement')
ax[1].imshow(dic_map.data.displacement[1,:,:],cmap='RdBu_r',vmin=vmin,vmax=vmax)
ax[1].set_title('v - displacement')
plt.tight_layout()


In [ ]:
# plotting points and lines on things 
# make a fake displacement field 
u = np.ones(dic_map.shape)*10
v = np.zeros(dic_map.shape)

vmin = -15
vmax = -vmin
fig,ax = plt.subplots(3,2)
ax[0,0].imshow(u,cmap='RdBu_r',vmin=vmin,vmax=vmax)
ax[0,0].set_title('u - displacement')
ax[0,1].imshow(v,cmap='RdBu_r',vmin=vmin,vmax=vmax)
ax[0,1].set_title('v - displacement')
plt.tight_layout()


# lets make up some coordinates 
# this should go diagonally bottom left to top right
start = [1000,1000] 
end = [2000,2000] 

for axs in ax[0,:]:
    axs.plot(start[0],start[1],'rx')
    axs.plot(end[0],end[1],'ro')
    axs.plot([start[0],end[0]],[start[1],end[1]],'r--')


# extract some profiles

# get u and v values 
profu = ski.measure.profile_line(u,np.flip(start),np.flip(end),linewidth=5)
profv = ski.measure.profile_line(v,np.flip(start),np.flip(end),linewidth=5)

ax[1,0].plot(profu)
ax[1,0].set_ylabel('Extracted profile / units')

ax[1,1].plot(profv)
# ax[1,1].set_ylabel('Extracted profile / units')


# resolve stuff onto line but using vectors 

# get line and tangent vectors
line_vect_dir = np.array(end) - np.array(start)
line_vect_dir = line_vect_dir/(np.sqrt(line_vect_dir[0]**2 + line_vect_dir[1]**2))

tang_vect_dir = np.array([-line_vect_dir[1],line_vect_dir[0]])

# get field vectors along line
uv_vect = np.array([profu,profv])

# resolve into components 
uv_tang_component = np.matmul(tang_vect_dir,uv_vect)
uv_line_component = np.matmul(line_vect_dir,uv_vect)

ax[2,0].set_title('components normal to line')
ax[2,0].plot(uv_tang_component)


ax[2,1].set_title('components parallel to line')
ax[2,1].plot(uv_line_component)

ax[2,0].set_xlabel('position / px')
ax[2,1].set_xlabel('position / px')

for axs in ax[1:,:].flatten().tolist():
    axs.set_ylim([vmin,vmax])

plt.tight_layout()